#### 2. Filtrage collaboratif basé sur l'utilisateur

 La deuxieme approche utilisé est un filtrage collaboratif(user_based_collaborative). La logique est de regrouper les utilsateurs par similarités. Et parmis pour recommander un film a un utilisateur, on utilise une moyenne pondérée deja noté par les utilisateurs proches de ce utilisateurs qui on déja regardé le film qu'on cherhce à recommander


## Similarité entre utilisateurs

**Cosinus** : angle entre les vecteurs de notes brutes.

$$\cos(u,v) = \frac{\sum_i r_{ui}\, r_{vi}}{\sqrt{\sum_i r_{ui}^2}\;\sqrt{\sum_i r_{vi}^2}}$$

**Pearson** : même chose, mais sur les notes centrées par la moyenne de chaque utilisateur.

$$\text{pearson}(u,v) = \frac{\sum_i (r_{ui}-\bar r_u)(r_{vi}-\bar r_v)}{\sqrt{\sum_i (r_{ui}-\bar r_u)^2}\;\sqrt{\sum_i (r_{vi}-\bar r_v)^2}}$$

Les sommes portent sur les films notés par les deux utilisateurs.

**Différence** : Pearson corrige le biais de notation (utilisateur généreux vs sévère), pas le cosinus. Pearson va de -1 à 1, le cosinus de 0 à 1 avec des notes positives.



 Pour notre cas on va utiliser la similarité cosinus comme mesure de proximité. Mais avant on va centrer la matrice utilisateur_film selon la moyenne de chaque utilisateur. Et donc ainsi on pourra remplacer les cases vides par 0 sans qu'il ny'ait un soucis d'interpretation. Une note de 0 quand tout est centré est comme si l'utilsateur navait rien noté

## Prédiction de la note (user-based)

$$\hat r_{ui} = \bar r_u + \frac{\sum_{v \in N_k(u)} \text{sim}(u,v)\,(r_{vi}-\bar r_v)}{\sum_{v \in N_k(u)} |\text{sim}(u,v)|}$$

| Terme | Signification |
|---|---|
| $\hat r_{ui}$ | note prédite pour l'utilisateur $u$ sur le film $i$ |
| $\bar r_u$ | note moyenne de l'utilisateur $u$ |
| $N_k(u)$ | les $k$ voisins les plus similaires à $u$ qui ont noté le film $i$ |
| $v$ | un voisin de $u$ |
| $\text{sim}(u,v)$ | similarité entre $u$ et $v$ (poids du voisin) |
| $r_{vi}$ | note donnée par le voisin $v$ au film $i$ |
| $\bar r_v$ | note moyenne du voisin $v$ |
| $r_{vi}-\bar r_v$ | écart du voisin : a-t-il aimé le film plus ou moins que d'habitude ? |
| $\sum \text{sim}(u,v)\,(r_{vi}-\bar r_v)$ | somme des écarts pondérés par la similarité |
| $\sum \lvert\text{sim}(u,v)\rvert$ | somme des poids, pour normaliser (moyenne pondérée) |

**Idée** : on part de la moyenne de $u$ et on lui ajoute l'écart moyen de ses voisins (pondéré par leur similarité) pour ce film.

NameError: name 'mat_user_film' is not defined

In [1]:
def noter(mat_user_film,moy_utilisateur,sim_user,k,film_non_vu,user):
    """ Cette fonction 
    
        mat_user: matrice utilisateur_film
        moy_utilisateur: vecteur des moyennes des notes des utilisateurs
        sim_user: matrice de similarité user x user
        k: le nombre de voisins proches à retourner
        film_non_vu: film dont on cherche à prévoir le score
        user: utilisateur à qui le film pourrait être proposé"""

    ##  Rechercher les utilisateurs qui ont noté le film
    noteurs = mat_user_film.index[mat_user_film[film_non_vu].notna()]

    ##  On cherche les utilisateurs les plus proches de notre individu parmis ceux qui ont noté le film
    sims = sim_user.loc[user, noteurs]

    ## on ne garde que les k utilisateurs proches ayant des similarités proches
    sims = sims[sims > 0]
    top = sims.nlargest(k)


    ## Si l'utilisateur n'a aucun voisin, la note prédite est juste sa moyenne
    if top.empty:
        return moy_utilisateur[user]
    ## On calcule maintenant le score à attribuer a notre cible en fonction des scores donnés par nos noteurs
    
    ## on calcule l'ecart de chaque voisin par rapport à sa moyenne 
    ecart = mat_user_film.loc[top.index, film_non_vu] - moy_utilisateur[top.index]

    ## on calcule la moyenne pondérée des ecarts
    moy_pond = (top * ecart).sum() / top.abs().sum()
    ## on ajoute le chiffre a la moyenne de notre cible
    pred = moy_utilisateur[user] + moy_pond

    return pred



In [4]:
## On va ecrire une fonction pour recommander n film a notre utilisateur
def recommander(utilisateur,nbre_film,mat_user_film,moy_utilisateur,sim_df,k=20):

    """
        nbre_film represente le nombre de films qu'on souhaite recommander
        sim_df represente la matrice de similarité cosinus
        moy_utilisateur represente le vecteur des notes moyennes des utilisateurs
        k represente le nombre d'utilisateur proches qu'on souhaite utiliser pour faire notre recomandation
    
    """
    films_non_regardes = mat_user_film.colums[mat_user_film.loc[utilisateur].isna()]
    taille = len(list(films_non_regardes))
    films_test = random.sample(list(films_non_regardes,k= min(700,taille)))

    ## on predit les scores
    film_id = []
    pred = []
    for i in films_test:
        score = noter(mat_user_film,moy_utilisateur,sim_df,k,i,utilisateur)
        film_id.append(i)
        pred.append(score)

    proposition = pd.DataFrame({"Films":film_id,"Score":pred})
    proposition = proposition.sort_values("Score",ascending=False).head(nbre_film)


    return proposition
        

In [ ]:
import pandas as pd

ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
    encoding="latin-1"
)


movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Type"],
    encoding="latin-1"
)

user_movie_matrix = ratings.pivot(
    index="UserID",
    columns="MovieID",
    values="Rating"
)

In [9]:
import pandas as pd
import random
from sklearn.metrics.pairwise import cosine_similarity


#On calcule la moyenne des notes des utilisateurs pour pouvoir remplacer les colonnes vides par 0, histoire qu'il n'yait pas de probleme. 
# On va centrer les notes et donc, si une note est supérieur à 0 elle est comptée positive, si inferieure a zero negative et zero si neutre
moy_utilisateur = user_movie_matrix.mean(axis=1) # vecteur des moyennes de chaque utilisateur

#  Centrer les notes, puis mettre 0 pour les films non notés
user_movie_matrix_centre = user_movie_matrix.sub(moy_utilisateur, axis=0).fillna(0)

#  Similarité cosinus entre tous les couples d'utilisateurs
sim_user = cosine_similarity(user_movie_matrix_centre)
sim_df = pd.DataFrame(sim_user, index=user_movie_matrix.index, columns=user_movie_matrix.index)

In [3]:
n_users = ratings["UserID"].nunique()
n_movies = ratings["MovieID"].nunique()
sparsity = 1 - len(ratings) / (n_users * n_movies)

print(f"la sparsité de la matrice est de {sparsity*100:.2f} %")


la sparsité de la matrice est de 95.53 %


In [6]:
## On va effectuer un test
## On va chercher les films qu'un utilisateur n'a pas vu et lui recommander

## prenons l'utilisateur 248
utilisateur = user_movie_matrix.index[248]
films_non_regardes = user_movie_matrix.columns[user_movie_matrix.loc[utilisateur].isna()]

print(f"les films non regardés pour notre utilisateur sont {films_non_regardes}")

les films non regardés pour notre utilisateur sont Index([   1,    2,    3,    4,    5,    6,    7,    8,    9,   11,
       ...
       3943, 3944, 3945, 3946, 3947, 3948, 3949, 3950, 3951, 3952],
      dtype='int64', name='MovieID', length=3596)


In [7]:
## On va selectionner 700 films au hasard qu'il n'a pas regardé
import random

random.seed(42)
films_test = random.sample(list(films_non_regardes),k=700)

In [10]:
## on predit les scores
film_id = []
pred = []
for i in films_test:
    score = noter(user_movie_matrix,moy_utilisateur,sim_df,20,i,248)
    film_id.append(i)
    pred.append(score)

resultat = {"Films":film_id,"Score":pred}
resultat = pd.DataFrame(resultat)

In [11]:
resultat.head()

,Films,Score
0,2915,4.086214
1,489,3.102362
2,111,4.062702
3,3368,4.463892
4,1255,4.022558


In [12]:
## On propose les 20 vingts films avec les meilleurs scores

proposition = resultat.sort_values("Score",ascending=False).head(20)
proposition.head()

,Films,Score
20,1039,5.295882
412,1212,5.226110
540,2127,5.165608
55,2762,5.162365
347,729,5.130964


In [13]:
## On veut les titres de ces films

final = pd.merge(proposition,movies[["MovieID","Title"]],left_on="Films",right_on="MovieID")
final

,Films,Score,MovieID,Title
0,1039,5.295882,1039,Synthetic Pleasures (1995)
1,1212,5.226110,1212,"Third Man, The (1949)"
2,2127,5.165608,2127,"First Love, Last Rites (1997)"
3,2762,5.162365,2762,"Sixth Sense, The (1999)"
4,729,5.130964,729,"Institute Benjamenta, or This Dream People Cal..."
5,2503,5.128844,2503,"Apple, The (Sib) (1998)"
6,1412,5.087141,1412,Some Mother's Son (1996)
7,2930,5.066251,2930,Return with Honor (1998)
8,669,5.055912,669,Aparajito (1956)
9,110,5.043738,110,Braveheart (1995)
